# 04b — Limpieza de Datos Físicos
Preprocesamiento basado en el TFG de referencia (Óscar Alcarria)  
Se aplica entre el notebook 04 (carga de medidas físicas) y el 05 (unión con red)

**Pasos:**
1. Eliminar sensores con valor constante (no aportan información)
2. Eliminar variables irrelevantes por conocimiento experto del dominio SWaT
3. Eliminar los primeros 100.000 registros del dataset Normal (periodo de arranque inestable)
4. Corregir el typo "A ttack" en Normal_Attack
5. Guardar de vuelta en los mismos paths para que el notebook 05 no necesite cambios

In [0]:
from pyspark.sql import functions as F

DELTA_NORMAL_PATH = "/Volumes/workspace/default/phisical_measures/delta_normal/"
DELTA_ATTACK_PATH = "/Volumes/workspace/default/phisical_measures/delta_attack/"

df_normal = spark.read.format("delta").load(DELTA_NORMAL_PATH)
df_attack = spark.read.format("delta").load(DELTA_ATTACK_PATH)

print(f"Normal — filas: {df_normal.count():,}  columnas: {len(df_normal.columns)}")
print(f"Attack — filas: {df_attack.count():,}  columnas: {len(df_attack.columns)}")

# Ver distribución de labels en attack para confirmar el typo
print("\n=== Valores de Normal_Attack en df_attack ===")
display(df_attack.groupBy("Normal_Attack").count().orderBy("count", ascending=False))

## 1 — Corregir typo 'A ttack'

In [0]:
# El dataset de ataque tiene un typo conocido: "A ttack" en lugar de "Attack"
df_attack = df_attack.withColumn(
    "Normal_Attack",
    F.when(F.col("Normal_Attack") == "A ttack", "Attack")
     .otherwise(F.col("Normal_Attack"))
)

print("Valores únicos en Normal_Attack tras corrección:")
display(df_attack.groupBy("Normal_Attack").count().orderBy("count", ascending=False))

## 2 — Identificar y eliminar sensores constantes

In [0]:
# Columnas a excluir del análisis de constantes
exclude_from_check = ["Timestamp", "timestamp_dt", "Normal_Attack"]

sensor_cols = [c for c in df_normal.columns if c not in exclude_from_check]
print(f"Total sensores/actuadores a analizar: {len(sensor_cols)}")

# Calcular número de valores distintos por columna en el dataset normal
# Un sensor constante tiene exactamente 1 valor distinto
distinct_counts = df_normal.select([
    F.countDistinct(c).alias(c) for c in sensor_cols
]).toPandas().T

distinct_counts.columns = ["n_distinct"]
distinct_counts = distinct_counts.sort_values("n_distinct")

print("\nSensores con valor constante (n_distinct == 1):")
constant_cols = distinct_counts[distinct_counts["n_distinct"] == 1].index.tolist()
print(constant_cols)
print(f"\nTotal a eliminar: {len(constant_cols)}")

## 3 — Eliminar variables irrelevantes por conocimiento experto

In [0]:
# Variables eliminadas basándose en conocimiento experto del dominio SWaT
# Son sensores analíticos de calidad del agua y caudalímetros no relevantes para detección
expert_drop = [
    "AIT201", "AIT202", "AIT203",
    "AIT401", "AIT402",
    "AIT501", "AIT502", "AIT503", "AIT504",
    "FIT301", "FIT401",
    "FIT501", "FIT502", "FIT503", "FIT504",
    "PIT501", "PIT502", "PIT503"
]

# Filtrar solo los que existen en el dataframe (por si alguno ya fue eliminado)
expert_drop_exist = [c for c in expert_drop if c in df_normal.columns]
print(f"Variables a eliminar por conocimiento experto: {len(expert_drop_exist)}")
print(expert_drop_exist)

In [0]:
# Combinar ambas listas de columnas a eliminar
all_drop = list(set(constant_cols + expert_drop_exist))

# Filtrar solo las que realmente existen en cada df
drop_normal = [c for c in all_drop if c in df_normal.columns]
drop_attack = [c for c in all_drop if c in df_attack.columns]

print(f"Total columnas a eliminar en df_normal: {len(drop_normal)}")
print(f"Total columnas a eliminar en df_attack: {len(drop_attack)}")
print(f"\nColumnas eliminadas:")
print(sorted(all_drop))

df_normal_clean = df_normal.drop(*drop_normal)
df_attack_clean = df_attack.drop(*drop_attack)

print(f"\ndf_normal: {len(df_normal.columns)} cols → {len(df_normal_clean.columns)} cols")
print(f"df_attack: {len(df_attack.columns)} cols → {len(df_attack_clean.columns)} cols")

## 4 — Eliminar periodo de arranque del dataset Normal

In [0]:
# El sistema tarda en estabilizarse desde el arranque.
# LIT101 tarda ~5.5h, AIT202/AIT203 hasta ~19.5h en alcanzar valores estables.
#  elimina los primeros 100.000 registros del conjunto normal.
# Solo aplica a df_normal — df_attack ya empieza con el sistema estabilizado.

filas_antes = df_normal_clean.count()

# Ordenar por timestamp y eliminar los primeros 100.000
from pyspark.sql.window import Window

w = Window.orderBy("timestamp_dt")
df_normal_clean = df_normal_clean     .withColumn("_row_num", F.row_number().over(w))     .filter(F.col("_row_num") > 100_000)     .drop("_row_num")

filas_despues = df_normal_clean.count()

print(f"Filas antes de eliminar arranque : {filas_antes:,}")
print(f"Filas eliminadas (arranque)      : {filas_antes - filas_despues:,}")
print(f"Filas después                    : {filas_despues:,}")

# Verificar rango temporal tras la eliminación
print("\n=== Rango temporal df_normal tras limpieza ===")
display(
    df_normal_clean.agg(
        F.min("timestamp_dt").alias("inicio"),
        F.max("timestamp_dt").alias("fin"),
        F.count("*").alias("total_filas")
    )
)

## 5 — Verificación final

In [0]:
print("=" * 55)
print("RESUMEN LIMPIEZA DATOS FÍSICOS")
print("=" * 55)
print(f"df_normal original  : {df_normal.count():,} filas, {len(df_normal.columns)} cols")
print(f"df_normal limpio    : {df_normal_clean.count():,} filas, {len(df_normal_clean.columns)} cols")
print(f"  - Arranque elim.  : 100.000 filas")
print(f"  - Cols eliminadas : {len(drop_normal)}")
print()
print(f"df_attack original  : {df_attack.count():,} filas, {len(df_attack.columns)} cols")
print(f"df_attack limpio    : {df_attack_clean.count():,} filas, {len(df_attack_clean.columns)} cols")
print(f"  - Typo corregido  : 'A ttack' → 'Attack'")
print(f"  - Cols eliminadas : {len(drop_attack)}")
print("=" * 55)

# Distribución de labels en ambos datasets limpios
print("\n=== Labels df_normal_clean ===")
display(df_normal_clean.groupBy("Normal_Attack").count().orderBy("Normal_Attack"))

print("\n=== Labels df_attack_clean ===")
display(df_attack_clean.groupBy("Normal_Attack").count().orderBy("Normal_Attack"))

## 6 — Guardar de vuelta en los mismos paths

In [0]:
# Sobreescribimos los mismos Deltas para que el notebook 05
# no necesite ningún cambio — lee de los mismos paths de siempre

df_normal_clean     .repartition(32)     .write     .format("delta")     .mode("overwrite")     .option("overwriteSchema", "true")     .save(DELTA_NORMAL_PATH)

print(f"✅ df_normal limpio guardado en: {DELTA_NORMAL_PATH}")

df_attack_clean     .repartition(32)     .write     .format("delta")     .mode("overwrite")     .option("overwriteSchema", "true")     .save(DELTA_ATTACK_PATH)

print(f"✅ df_attack limpio guardado en: {DELTA_ATTACK_PATH}")

# Verificación rápida
df_n_check = spark.read.format("delta").load(DELTA_NORMAL_PATH)
df_a_check = spark.read.format("delta").load(DELTA_ATTACK_PATH)

print(f"\nVerificación lectura:")
print(f"  normal  → {df_n_check.count():,} filas, {len(df_n_check.columns)} cols")
print(f"  attack  → {df_a_check.count():,} filas, {len(df_a_check.columns)} cols")